# 2D Try-On Experiments

Face analysis, segmentation, recommendation, and a few try-on renders.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from systems.static_auto_tryon.auto_app.core.asset_bank import asset_bank_summary, get_asset_by_id
from systems.static_auto_tryon.auto_app.core.face_analyzer import analyze_face_image
from systems.static_auto_tryon.auto_app.core.hair_segmentation import predict_hair_mask_image, save_predicted_hair_mask
from systems.static_auto_tryon.auto_app.core.recommender import recommend_hairstyles
from systems.static_auto_tryon.auto_app.core.tryon_2d_engine import render_static_tryon, render_static_tryon_image
from systems.static_auto_tryon.auto_app.models.schemas import FaceAttributes, RecommendationPreferences

PROJECT_ROOT


In [ ]:
import importlib
import systems.static_auto_tryon.auto_app.core.tryon_2d_engine as tryon_2d_engine

importlib.reload(tryon_2d_engine)
from systems.static_auto_tryon.auto_app.core.tryon_2d_engine import render_static_tryon, render_static_tryon_image

print('Renderer module:', tryon_2d_engine.__name__)
print('Default renderer:', render_static_tryon.__name__)


In [ ]:
INPUT_IMAGE_PATH = PROJECT_ROOT / 'backend' / 'outputs' / 'uploads' / 'image4.png'
TARGET_GENDER = 'female'
TOP_K = 5
TRYON_TOP_K = 3

INPUT_IMAGE_PATH

In [ ]:
bank = asset_bank_summary()
print('Active asset bank:', bank['metadata_dir'])
print('Asset count:', bank['asset_count'])
bank

In [ ]:
if not INPUT_IMAGE_PATH.exists():
    raise FileNotFoundError(f'Image not found: {INPUT_IMAGE_PATH}')

analysis = analyze_face_image(INPUT_IMAGE_PATH)
print('Face detected:', analysis.face_detected)
print('Message:', analysis.message)
analysis.geometry, analysis.face_attributes

In [ ]:
with Image.open(INPUT_IMAGE_PATH) as source_image:
    subject_hair_mask = predict_hair_mask_image(source_image)

mask_path = save_predicted_hair_mask(subject_hair_mask, INPUT_IMAGE_PATH.stem)
print('Saved mask to:', mask_path)
mask_path

In [ ]:
preferences = RecommendationPreferences(
    allow_bangs=True,
    target_gender=TARGET_GENDER,
)
face_attrs = FaceAttributes(**analysis.face_attributes)
recommendation_response = recommend_hairstyles(face_attrs, preferences=preferences, top_k=TOP_K)
recommendation_rows = [
    {
        'asset_id': item.asset_id,
        'score': item.score,
        'gender_suitability': item.gender_suitability,
        'length': item.normalized_attributes.length,
        'curl': item.normalized_attributes.curl,
        'style_family': item.normalized_attributes.style_family,
        'reason': item.reason,
        'image_path': item.image_path,
    }
    for item in recommendation_response.recommendations
]
recommendation_df = pd.DataFrame(recommendation_rows)
recommendation_df

In [ ]:
tryon_rows = []
for item in recommendation_response.recommendations[:TRYON_TOP_K]:
    asset = get_asset_by_id(item.asset_id)
    if asset is None:
        continue
    output_path = render_static_tryon(
        INPUT_IMAGE_PATH,
        analysis,
        asset,
        subject_hair_mask=subject_hair_mask,
    )
    tryon_rows.append(
        {
            'asset_id': item.asset_id,
            'score': item.score,
            'style_family': item.normalized_attributes.style_family,
            'renderer': 'default',
            'output_path': output_path,
        }
    )

tryon_df = pd.DataFrame(tryon_rows)
tryon_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
with Image.open(INPUT_IMAGE_PATH) as input_image:
    axes[0].imshow(input_image.convert('RGB'))
axes[0].set_title('Input Image')
axes[0].axis('off')
axes[1].imshow(subject_hair_mask, cmap='gray')
axes[1].set_title('Predicted Hair Mask')
axes[1].axis('off')
plt.tight_layout()

In [ ]:
if tryon_rows:
    fig, axes = plt.subplots(len(tryon_rows), 1, figsize=(6, 4 * len(tryon_rows)))
    if len(tryon_rows) == 1:
        axes = [axes]
    for axis, row in zip(axes, tryon_rows):
        with Image.open(row['output_path']) as output_image:
            axis.imshow(output_image.convert('RGB'))
        axis.set_title(f"{row['asset_id']} | {row['style_family']} | score={row['score']:.4f}")
        axis.axis('off')
    plt.tight_layout()
else:
    print('No try-on outputs were created.')

## Current Static Renderer Reference

The old MediaPipe-first experimental renderer was removed during cleanup. This section now keeps the notebook aligned with the current static auto try-on system only.


In [ ]:
mediapipe_tryon_rows = tryon_rows.copy()
mediapipe_tryon_df = pd.DataFrame(mediapipe_tryon_rows)
print('Experimental MediaPipe renderer is no longer part of the current project structure.')
mediapipe_tryon_df


In [ ]:
if mediapipe_tryon_rows:
    fig, axes = plt.subplots(len(mediapipe_tryon_rows), 1, figsize=(6, 4 * len(mediapipe_tryon_rows)))
    if len(mediapipe_tryon_rows) == 1:
        axes = [axes]
    for axis, row in zip(axes, mediapipe_tryon_rows):
        with Image.open(row['output_path']) as output_image:
            axis.imshow(output_image.convert('RGB'))
        axis.set_title(f"{row['asset_id']} | {row['style_family']} | score={row['score']:.4f} | Current static renderer")
        axis.axis('off')
    plt.tight_layout()
else:
    print('No current renderer outputs were created.')


In [ ]:
comparison_df = tryon_df.copy()
comparison_df
